# BB: logical-memory simulation

Prepare logical memory in the selected basis, apply noisy syndrome-extraction
cycles, and decode the final logical observables. The circuit noise satisfies
$p_{\mathrm{CX}}=p_{\mathrm{measurement}}=p_{\mathrm{reset}}=p$, including
final data measurement. No idle-noise channel is included.

`generate` writes circuit artifacts and detector/observable samples; `decode`
uses saved samples; `both` performs both stages. The `xz` mode retains the
CSS detector type relevant to the memory basis, while `xyz` retains all types.

Check `SHOTS` before running: these settings use large sample counts.
Outputs go to `bivariate_bicycle/results/`.


## Parameters


In [ ]:
import sys
from pathlib import Path

REPO_ROOT = next(
    path
    for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "code_construction" / "affine_codes.py").is_file()
)
sys.path.insert(0, str(REPO_ROOT))

In [ ]:
import os
import shlex
import subprocess
import sys
from pathlib import Path

PYTHON = Path(sys.executable)
SCRIPT = REPO_ROOT / "bivariate_bicycle/simulate_memory.py"
SAMPLE_DIR = Path("bivariate_bicycle/results/samples")
DECODE_DIR = Path("bivariate_bicycle/results/decode")

STAGE = "both"
# Code and physical noise.
CODES = ["bb_288_12_18"]
BASIS = "X"  # Z | X | both
DECODING_MODE = "xz"  # xyz | xz | both
P_LIST = [0.0045]
CYCLES = 8
SHOTS = 2_000_000
SHOT_CHUNK = 200
WORKERS = max(1, min(8, os.cpu_count() or 1))
SEED = 20260704
DECODE_BATCH_SIZE = 100000
PROGRESS_SECONDS = 30

# Primary RelayBP decoder.
GAMMA0 = 0.1
PRE_ITER = 200
NUM_SETS = 20
SET_MAX_ITER = 100
GAMMA_DIST_MIN = -0.24
GAMMA_DIST_MAX = 0.6
STOP_NCONV = 1

# Second RelayBP pass on unresolved shots.
RELAYBP_FALLBACK = True
RELAYBP_FALLBACK_RETRY_UNCONVERGED = False
RELAYBP_FALLBACK_SOURCE_HASH = ""
RELAYBP_FALLBACK_GAMMA0 = 0.0
RELAYBP_FALLBACK_PRE_ITER = 1000
RELAYBP_FALLBACK_NUM_SETS = 200
RELAYBP_FALLBACK_SET_MAX_ITER = 400
RELAYBP_FALLBACK_GAMMA_DIST_MIN = -0.24
RELAYBP_FALLBACK_GAMMA_DIST_MAX = 0.6
RELAYBP_FALLBACK_STOP_NCONV = 1

FORCE_ARTIFACTS = False
FORCE_SHOTS = False
FORCE_DECODE = False

## Command


In [ ]:
def _csv(values):
    return ",".join(str(value) for value in values)


cmd = [
    str(PYTHON),
    str(SCRIPT),
    "--stage",
    STAGE,
    "--codes",
    _csv(CODES),
    "--basis",
    BASIS,
    "--decoding-mode",
    DECODING_MODE,
    "--p-list",
    _csv(P_LIST),
    "--cycles",
    str(CYCLES),
    "--shots",
    str(SHOTS),
    "--shot-chunk",
    str(SHOT_CHUNK),
    "--sample-dir",
    str(SAMPLE_DIR),
    "--decode-dir",
    str(DECODE_DIR),
    "--workers",
    str(WORKERS),
    "--seed",
    str(SEED),
    "--decode-batch-size",
    str(DECODE_BATCH_SIZE),
    "--progress-seconds",
    str(PROGRESS_SECONDS),
    "--gamma0",
    str(GAMMA0),
    "--pre-iter",
    str(PRE_ITER),
    "--num-sets",
    str(NUM_SETS),
    "--set-max-iter",
    str(SET_MAX_ITER),
    "--gamma-dist-min",
    str(GAMMA_DIST_MIN),
    "--gamma-dist-max",
    str(GAMMA_DIST_MAX),
    "--stop-nconv",
    str(STOP_NCONV),
]

if RELAYBP_FALLBACK:
    cmd.append("--relaybp-fallback")
    cmd.extend(["--relaybp-fallback-gamma0", str(RELAYBP_FALLBACK_GAMMA0)])
    cmd.extend(["--relaybp-fallback-pre-iter", str(RELAYBP_FALLBACK_PRE_ITER)])
    cmd.extend(["--relaybp-fallback-num-sets", str(RELAYBP_FALLBACK_NUM_SETS)])
    cmd.extend(["--relaybp-fallback-set-max-iter", str(RELAYBP_FALLBACK_SET_MAX_ITER)])
    cmd.extend(["--relaybp-fallback-gamma-dist-min", str(RELAYBP_FALLBACK_GAMMA_DIST_MIN)])
    cmd.extend(["--relaybp-fallback-gamma-dist-max", str(RELAYBP_FALLBACK_GAMMA_DIST_MAX)])
    cmd.extend(["--relaybp-fallback-stop-nconv", str(RELAYBP_FALLBACK_STOP_NCONV)])
    if RELAYBP_FALLBACK_RETRY_UNCONVERGED:
        cmd.append("--relaybp-fallback-retry-unconverged")
        if str(RELAYBP_FALLBACK_SOURCE_HASH).strip():
            cmd.extend(
                ["--relaybp-fallback-source-hash", str(RELAYBP_FALLBACK_SOURCE_HASH).strip()]
            )

if FORCE_ARTIFACTS:
    cmd += ["--force-artifacts"]
if FORCE_SHOTS:
    cmd += ["--force-shots"]
if FORCE_DECODE:
    cmd += ["--force-decode"]

print(" ".join(shlex.quote(part) for part in ["python", *cmd[1:]]))

## Run


In [ ]:
completed = subprocess.run(cmd, cwd=str(REPO_ROOT), text=True)
if completed.returncode:
    raise SystemExit(completed.returncode)